In [ ]:
import os
import time
from pathlib import Path
import jax
import jax.numpy as jp
import mujoco
from mujodico import mjx
import numpy as np
import matplotlib.pyplot as plt
import aligator
from aligator import constraints, manifolds, dynamics
import viser
from mjviser import ViserMujocoScene

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
path = Path.cwd() / "1d-pusher.xml"
model = mujoco.MjModel.from_xml_path(str(path))

mjx_smooth_model = mjx.put_model(
    model,
    contact_approximation="log barrier",
    contact_smoothing=1e1,
)

mjx_model = mjx.put_model(
    model,
    contact_approximation="log barrier",
    contact_smoothing=0.0,
)

mjx_data = mjx.make_data(mjx_model)

nq = model.nq
nv = model.nv
nx = nq + nv
nu = model.nu

space = manifolds.VectorSpace(nx)

class PusherDynamics(dynamics.ExplicitDynamicsModel):
    def __init__(self):
        super().__init__(space, nu)

        def step_fn(x, u):
            d = mjx_data.replace(qpos=x[:nq], qvel=x[nq:], ctrl=u)
            d = mjx.step(mjx_model, d)
            return jp.concatenate((d.qpos, d.qvel))

        def smooth_step_fn(x, u):
            d = mjx_data.replace(qpos=x[:nq], qvel=x[nq:], ctrl=u)
            d = mjx.step(mjx_smooth_model, d)
            return jp.concatenate((d.qpos, d.qvel))

        self._step = jax.jit(step_fn)
        self._jac = jax.jit(jax.jacfwd(smooth_step_fn, argnums=(0, 1)))

    def forward(self, x, u, data):
        xnext = self._step(jp.asarray(x), jp.asarray(u))
        data.xnext[:] = np.asarray(xnext, dtype=np.float64)

    def dForward(self, x, u, data):
        A, B = self._jac(jp.asarray(x), jp.asarray(u))
        A = np.asarray(A, dtype=np.float64)
        B = np.asarray(B, dtype=np.float64)
        if np.isnan(A).any() or np.isnan(B).any():
            raise RuntimeError("NaN in MJX dynamics Jacobians")
        data.Jx[:, :] = A
        data.Ju[:] = B.reshape(data.Ju.shape)

In [ ]:
dyn_model = PusherDynamics()

N = 200

x0 = np.array([0.0, 0.0, 0.0, 0.0])
x_goal = np.array([0.0, 1.0, 0.0, 0.0])

Q = np.diag([0.0, 1.0, 0.0, 1.0])
Qf = np.diag([0.0, 100.0, 0.0, 10.0])
R = np.ones(nu) * 0.10

running_cost = aligator.CostStack(space, nu)

running_cost.addCost(
    aligator.QuadraticStateCost(space, nu, x_goal, Q),
)
running_cost.addCost(
    aligator.QuadraticControlCost(space, np.zeros(nu), R),
)

terminal_cost = aligator.CostStack(space, nu)
terminal_cost.addCost(
    aligator.QuadraticStateCost(space, nu, x_goal, Qf),
)

problem = aligator.TrajOptProblem(x0, nu, space, terminal_cost)

u_min = -1.0 * np.ones(nu)
u_max =  1.0 * np.ones(nu)
ctrl_res = aligator.ControlErrorResidual(nx, nu)

for _ in range(N):
    stage = aligator.StageModel(running_cost, dyn_model)
    stage.addConstraint(ctrl_res, constraints.BoxConstraint(u_min, u_max))
    problem.addStage(stage)

us_init = [np.zeros(nu) for _ in range(N)]
xs_init = [x0 for _ in range(N+1)]

solver = aligator.SolverProxDDP(
    tol=1e-3,
    max_iters=15,
    verbose=aligator.VerboseLevel.VERBOSE,
)

solver.setup(problem)
solver.run(problem, xs_init, us_init)

res = solver.results

xs_opt = np.asarray(res.xs)
us_opt = np.asarray(res.us)

print(res)
print("Initial state:", x0)
print("Goal state:", x_goal)
print("Final state:", xs_opt[-1])

In [ ]:
server = viser.ViserServer()
server.scene.add_grid("grid")
scene = ViserMujocoScene(server, model, num_envs=1)
scene.camera_tracking_enabled = False

data = mujoco.MjData(model)
mujoco.mj_kinematics(model, data)
scene.update_from_mjdata(data)

In [ ]:
mjx_model = mjx.put_model(model, contact_approximation="mujoco")
mjx_data = mjx.make_data(mjx_model)

@jax.jit
def step(x, u):
    d = mjx_data.replace(qpos=x[:nq], qvel=x[nq:], ctrl=u)
    d = mjx.step(mjx_model, d)
    return jp.concatenate((d.qpos, d.qvel))

xs_traj = jp.zeros_like(xs_opt)
xs_traj = xs_traj.at[0].set(x0)
for k in range(N):
    tic = time.time()
    x_kp1 = step(xs_traj[k], jp.array(us_opt[k]))
    xs_traj = xs_traj.at[k+1].set(x_kp1)

    data.qpos = np.array(x_kp1[:nq])
    mujoco.mj_kinematics(model, data)
    scene.update_from_mjdata(data)

    remain = model.opt.timestep - (time.time() - tic)
    if remain > 0:
        time.sleep(remain)

ts = np.arange(N + 1) * model.opt.timestep
plt.plot(ts, xs_opt[:, 0], 'r--', label='pusher reference')
plt.plot(ts, xs_opt[:, 1], 'b--', label='object reference')
plt.plot(ts, xs_traj[:, 0], 'r-', label='pusher rollout')
plt.plot(ts, xs_traj[:, 1], 'b-', label='object rollout')
plt.axhline(x_goal[1], color='tab:blue', linestyle=':', label='object goal')
plt.xlabel('Time')
plt.ylabel('Position')
plt.xlim([min(ts), max(ts)])
plt.legend()
plt.show()